# Grasping in the Dark — watch RL teach a robot arm to grasp 🦾
### HIL-SERL (SAC + RLPD) in simulation, on the LeRobot `gym-hil` Franka Panda

This is the hands-on notebook for **Session 3 (Robotics)** of the *RL in Production* workshop
(Vizuara AI Labs). You will watch **reinforcement learning** teach a Franka Panda to **pick up and
lift a cube** — learning purely from a **sparse reward** it initially almost never receives, no human
demonstrations of the final skill required.

### Two ways to use this notebook

| | What you do | Where it runs | Time |
|---|---|---|---|
| **A · Run & understand** *(this notebook)* | Load the trained policy, watch it grasp, measure its success rate, read the learning curve, and understand *how* SAC+RLPD works. | **Colab, free T4 GPU** | ~10 min |
| **B · Train it yourself from scratch** *(Section 7)* | Reproduce the 0% → ~100% training run on a real GPU. | **Modal / your own GPU box** (not Colab) | ~40 min |
| **C · Put it on a real arm** *(Section 8)* | Bring the exact same stack up on a physical **$200 SO-101**. | Your desk | ~1 day |

**Path A is fully self-contained — you need nothing but this notebook.** Start there. Sections 7 and 8
give you everything to go further.

> **Runtime:** set **Runtime → Change runtime type → T4 GPU** before you begin.

Code, checkpoint, paper, and website: [https://github.com/VizuaraAI/RL-in-Production-Bootcamp-Resources/tree/main/phase-2/grasping-in-the-dark](https://github.com/VizuaraAI/RL-in-Production-Bootcamp-Resources/tree/main/phase-2/grasping-in-the-dark).

## 0 · Setup — GPU + headless MuJoCo + install

MuJoCo (the physics simulator) needs an OpenGL backend to render the camera images. On Colab's Linux
GPU the right one is **EGL**, and it **must be selected *before* MuJoCo is imported** — so we set it
in the very first cells. Then we install `gym-hil` (the sim) and LeRobot with the HIL-SERL extra
(the SAC/RLPD learner stack).

In [ ]:
!nvidia-smi -L || echo 'No GPU — set Runtime → Change runtime type → T4 GPU, then rerun.'

In [ ]:
import os, subprocess
# Headless EGL for MuJoCo offscreen rendering on Colab's NVIDIA GPU.
subprocess.run('apt-get -qq install -y libgl1-mesa-glx libglfw3 libglew2.2 libegl1-mesa-dev libgles2-mesa-dev ffmpeg', shell=True)
p = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
os.makedirs(os.path.dirname(p), exist_ok=True)
if not os.path.exists(p):
    open(p, 'w').write('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')
os.environ['MUJOCO_GL'] = 'egl'          # MUST be set before importing mujoco/gym_hil
os.environ['PYOPENGL_PLATFORM'] = 'egl'
print('EGL configured')

In [ ]:
# Install gym-hil (the sim) + LeRobot with the HIL-SERL extra (SAC/RLPD stack).
# Heavy deps (mujoco, grpcio, torch). If Colab pops 'RESTART SESSION' after this cell:
#   click it, then Runtime → Run all — the install is cached, so it's fast the second time.
!pip -q install gym-hil imageio imageio-ffmpeg
!git clone -q --depth 1 https://github.com/huggingface/lerobot /content/lerobot
!pip -q install -e '/content/lerobot[hilserl]'
print('installed — if Colab asked you to restart, do it, then Run all')

## 1 · The task — `PandaPickCube`

A 7-DoF **Franka Panda** must **lift a cube by more than 10 cm**. This is the whole task — but the way
the agent perceives and is rewarded is what makes it a genuine RL problem:

| | |
|---|---|
| **Observation** | two **128×128 RGB** camera images (a *front* view + a *wrist* view) **+** an **18-D** proprioceptive state (`qpos` ×7, `qvel` ×7, gripper ×1, end-effector position ×3) |
| **Action** | a **3-D end-effector displacement** (Δx, Δy, Δz) **+** a **discrete gripper** command (open / close) |
| **Reward** | **sparse**: `1.0` the instant the cube is lifted >10 cm, `0.0` every other step |

That sparse reward is the crux. The agent gets **no gradient signal at all** until it *accidentally*
completes a full reach → grasp → lift. Discovering that by random exploration is a needle in a
haystack — which is exactly why the prior-data trick (RLPD, Section 6) matters so much.

In [ ]:
import os
os.environ.setdefault('MUJOCO_GL', 'egl')
import numpy as np, gymnasium as gym, gym_hil  # importing gym_hil registers the gym_hil/* env ids

env = gym.make('gym_hil/PandaPickCubeBase-v0', image_obs=True)
obs, info = env.reset(seed=0)
print('observation keys :', list(obs.keys()))
print('agent_pos (state):', np.asarray(obs['agent_pos']).shape,   '(qpos7 + qvel7 + gripper1 + tcp3 = 18)')
print('front / wrist img :', obs['pixels']['front'].shape, '/', obs['pixels']['wrist'].shape)
print('action space     :', env.action_space)
assert np.asarray(obs['agent_pos']).shape == (18,)
assert obs['pixels']['front'].shape == (128, 128, 3)

In [ ]:
# Peek at what the robot sees (front | wrist).
import matplotlib.pyplot as plt
panel = np.concatenate([obs['pixels']['front'], obs['pixels']['wrist']], axis=1)
plt.figure(figsize=(6, 3)); plt.imshow(panel); plt.axis('off'); plt.title('front  |  wrist'); plt.show()

## 2 · The seed demonstrations — where prior data comes from

RLPD makes **every training batch 50% online experience and 50% prior demonstrations**. Those prior
demonstrations are what keep *successful* (state, action) pairs in front of the critic before the
policy can succeed on its own.

**Where do they come from?** For this sim task we reuse the **public** dataset
[`lilkm/pick_cube_franka_panda_30`](https://huggingface.co/datasets/lilkm/pick_cube_franka_panda_30)
(~30 teleoperated episodes) — the standard seed set shipped with the LeRobot `gym-hil` HIL-SERL
example. **We did not collect these ourselves.** On a real robot (Section 8) you would record ~15–25
of your own teleop demos instead.

To build intuition for *what a demonstration is*, here's a **hand-scripted** reach-grasp-lift — a plain
controller, **not** the RL policy (that's Section 3). It's the kind of trajectory the offline buffer is
seeded with.

In [ ]:
from IPython.display import HTML
import imageio, base64

def cube_and_tcp(env, obs):
    tcp = np.asarray(obs['agent_pos']).reshape(-1)[-3:]
    data = getattr(env.unwrapped, 'data', None) or getattr(env.unwrapped, '_data')
    cube = np.asarray(data.body('block').xpos).reshape(-1)[:3]
    return cube, tcp

def scripted_action(env, obs):
    cube, tcp = cube_and_tcp(env, obs); d = cube - tcp
    a = np.zeros(env.action_space.shape, dtype=np.float32)
    if np.linalg.norm(d[:2]) > 0.03:      a[:2] = np.clip(d[:2]*4, -.05, .05); a[2] = np.clip(d[2]*2, -.02, .03); a[-1] = -1.0
    elif d[2] < -0.01:                    a[2] = np.clip(d[2]*4, -.05, 0.0); a[-1] = -1.0
    else:                                 a[-1] = 1.0; a[2] = 0.04
    return a

def rollout_frames(policy_fn, seed=100, max_steps=100):
    o, _ = env.reset(seed=seed); frames=[]; succ=False
    for _ in range(max_steps):
        frames.append(np.concatenate([o['pixels']['front'], o['pixels']['wrist']], axis=1))
        o, r, term, trunc, info = env.step(policy_fn(env, o))
        succ = succ or bool(info.get('succeed'))
        if term or trunc: break
    return frames, succ

def show_video(frames, fps=12):
    imageio.mimsave('/tmp/clip.mp4', frames, fps=fps, quality=8)
    b64 = base64.b64encode(open('/tmp/clip.mp4','rb').read()).decode()
    return HTML(f'<video autoplay loop controls width=420 src="data:video/mp4;base64,{b64}">')

frames, ok = rollout_frames(scripted_action, seed=100)
print('scripted grasp success:', ok, '| frames:', len(frames))
show_video(frames)

## 3 · The trained HIL-SERL policy — load it from the Hub

This policy was trained with **SAC + RLPD** on a single **L4 GPU** (~40 min) — an actor and a learner
running asynchronously over gRPC, learning from the environment's sparse reward. We now download the
published checkpoint and load it.

### ⚠️ The one thing everyone gets wrong: normalization

`from_pretrained` loads the **network weights only**. `GaussianActorPolicy.select_action` does **no**
normalization internally — it *assumes* the observation is already normalized and returns a `tanh`
action in `[-1, 1]`. So if you feed it raw observations and use the raw action, **the gripper never
closes and the policy scores ~0% — even though it trained to ~100%.**

The fix: load the checkpoint's **saved processors** and wrap every step —
`normalize observation → select_action → un-normalize action` — exactly as the training actor did.
That's what `make_pre_post_processors(pretrained_path=...)` gives you.

In [ ]:
CKPT_REPO = 'RajatDandekar/hilserl-panda-pickcube-sac'   # <- the published HIL-SERL checkpoint (public; a pretrained_model dir)

import torch, draccus
from huggingface_hub import snapshot_download
from lerobot.rl.train_rl import TrainRLServerPipelineConfig
from lerobot.rl import gym_manipulator as gm
from lerobot.processor import TransitionKey
from lerobot.policies.gaussian_actor.modeling_gaussian_actor import GaussianActorPolicy
from lerobot.policies import make_pre_post_processors

device = 'cuda' if torch.cuda.is_available() else 'cpu'
local = snapshot_download(CKPT_REPO)                # downloads weights + processors + train_config.json + train.log
cfg = draccus.parse(TrainRLServerPipelineConfig, args=['--config_path', f'{local}/train_config.json'])
cfg.env.task = 'PandaPickCube-v0'          # autonomous, headless (no human, no input device)
policy = GaussianActorPolicy.from_pretrained(local).to(device).eval()
eval_env, teleop = gm.make_robot_env(cfg.env)
env_proc, act_proc = gm.make_processors(eval_env, teleop, cfg.env, device)
input_keys = list(cfg.policy.input_features.keys())

# Load the checkpoint's saved normalizer / un-normalizer (see the note above).
pre, post = make_pre_post_processors(policy_cfg=cfg.policy, pretrained_path=local)
n_disc = getattr(cfg.policy, 'num_discrete_actions', None)

def act(obs_full):
    with torch.no_grad():
        a = policy.select_action(batch=pre.process_observation(obs_full))   # normalize obs in
    if n_disc is not None:                       # un-normalize the continuous dims; keep the discrete gripper bit as-is
        cont = post.process_action(a[..., :-1])
        return torch.cat([cont, a[..., -1:].to(cont.device)], dim=-1)
    return post.process_action(a)                # un-normalize action out

print('loaded policy on', device, '| inputs:', input_keys, '| normalizers loaded \u2713')

## 4 · Watch it grasp — and measure the success rate

Now we roll the policy out autonomously across randomized cube positions, record the first episode as
video, and report the success rate. A converged checkpoint grasps essentially every time.

In [ ]:
def eval_policy(n_episodes=20, record_first=True):
    successes, frames = [], []
    for ep in range(n_episodes):
        tr = gm.reset_and_build_transition(eval_env, env_proc, act_proc)
        while True:
            action = act(tr[TransitionKey.OBSERVATION])   # normalize \u2192 select_action \u2192 un-normalize
            tr = gm.step_env_and_process_transition(env=eval_env, transition=tr, action=action,
                                                    env_processor=env_proc, action_processor=act_proc)
            if record_first and ep == 0:
                r = eval_env.render(); r = r[0] if isinstance(r, (list, tuple)) else r
                frames.append(np.asarray(r).astype(np.uint8))
            reward = float(tr[TransitionKey.REWARD])
            if bool(tr[TransitionKey.DONE]) or bool(tr[TransitionKey.TRUNCATED]):
                successes.append(1.0 if reward > 0 else 0.0); break
    print(f'success rate over {n_episodes} episodes: {np.mean(successes):.0%}')
    return successes, frames

_, frames = eval_policy(n_episodes=20)
show_video(frames) if frames else None

## 5 · How it learned — the reward curve *and why it's a phase transition*

The training log is published **alongside the checkpoint**. Let's plot the learning curve, then read it
carefully — because it is not a smooth ramp, it's a **phase transition**, and understanding *why* is the
most instructive part of this whole exercise.

In [ ]:
# The training log ships in the checkpoint repo. Parse 'Global step N: Episode reward: R'.
import re, glob
logs = glob.glob(f'{local}/**/train.log', recursive=True) + glob.glob(f'{local}/train.log')
pts = []
if logs:
    txt = open(logs[0]).read()
    pts = [(int(s), float(r)) for s, r in re.findall(r'Global step (\d+): Episode reward: ([-\d.]+)', txt)]
if pts:
    xs, ys = zip(*pts)
    w = max(1, len(ys)//25); sm = np.convolve(ys, np.ones(w)/w, 'valid')
    plt.figure(figsize=(7,3)); plt.plot(xs, ys, alpha=.25, label='episode reward')
    plt.plot(xs[w-1:], sm, lw=2, label=f'moving avg ({w})')
    plt.xlabel('actor env step'); plt.ylabel('reward'); plt.legend(); plt.title('HIL-SERL learning curve'); plt.show()
else:
    print('No train.log found in the checkpoint repo.')

In [ ]:
# Anatomy of the run: locate the first success, then the success rate in the early vs late phase.
if pts:
    succ = [s for s, r in pts if r > 0]
    first = succ[0] if succ else None
    def rate(lo, hi):
        w = [1.0 if r > 0 else 0.0 for s, r in pts if lo <= s < hi]
        return (sum(w) / len(w)) if w else float('nan')
    last = xs[-1]
    print(f'first successful grasp     : ~step {first}')
    print(f'success rate, first third  : {rate(0, last/3):.0%}   <- the "dark phase": reward almost never arrives')
    print(f'success rate, last third   : {rate(2*last/3, last+1):.0%}   <- consolidation: the policy has locked in')

**Read the three regimes in the numbers above:**

1. **The dark phase** — reward almost never arrives, so it *looks* like nothing is happening. But because
   half of every batch is offline **successes**, the **critic is quietly building a value landscape** the
   whole time — it learns that "gripper closing on the cube, cube rising" is worth a lot, long before the
   actor can produce that state.
2. **Discovery** — the first autonomous success isn't the *start* of learning; it's the moment the actor
   finally stumbles into a region the critic **already prizes**. That success lands on fertile ground and
   is reinforced immediately, instead of being diluted into noise.
3. **Consolidation** — online and offline data now agree, entropy anneals, exploration narrows, and the
   success rate saturates near 100%.

The one design choice that makes this work is RLPD's **50/50 sampling**: one success buried in ~2,000
online failures is a signal-to-noise ratio of ~1:2000 — invisible. Forcing half of every batch to be
known successes raises it to ~1:1. That's the whole trick.

## 6 · How HIL-SERL actually works (the pieces)

What you just ran is the **learning core** of HIL-SERL. The full method stacks four ideas:

1. **SAC** *(Soft Actor-Critic)* — a sample-efficient, off-policy, maximum-entropy actor-critic. "Off-policy"
   is the load-bearing word: it learns from a **replay buffer** of past transitions, not just fresh
   rollouts, which is what makes it cheap enough for real hardware.
2. **RLPD** *(RL with Prior Data)* — the sample-efficiency engine: **50/50 online/offline sampling**,
   **LayerNorm on the critics** (which stabilizes bootstrapping from off-distribution data), and a high
   update-to-data ratio. This is *why* a sparse-reward grasp is learnable in thousands, not millions, of steps.
3. **A learned reward from pixels** — on a **real robot** there is no simulator to hand you the reward, so
   HIL-SERL trains a small **binary success classifier** on camera frames. **In simulation we skip this
   entirely** — the MuJoCo environment computes the reward from ground-truth state, so training is fully
   autonomous. (This is why *this* notebook needs no human and no classifier.)
4. **Human-in-the-loop interventions** — on real hardware a person watches the policy and takes over with a
   gamepad the instant it's about to fail; those corrections enter the replay buffer and the intervention
   rate **fades to zero** as the policy takes over. That interface needs a physical controller, so it can't
   run in a headless notebook — see the clip on the [project page](https://hil-serl.github.io).

> **The simulation simplification in one line:** in sim, pieces 3 and 4 disappear at runtime — the human is
> a *concept we teach*, not a dependency — and what's left is exactly the SAC + RLPD core you evaluated above.

## 7 · Train it yourself from scratch (Path B)

Everything above *used* a trained checkpoint. Here is how that checkpoint was **produced** — how you get
your own 0% → ~100% run.

### Why not in this Colab?

Full training runs an **asynchronous actor + learner as two processes over gRPC** for ~40 minutes, with
frequent checkpointing (cloud GPUs get preempted near convergence). A free Colab session times out, is
single-process-friendly, and doesn't persist checkpoints — so training belongs on a **persistent GPU**.
Two good options follow. (Even the original HIL-SERL is one-GPU-for-hours — never a notebook.)

### Option 1 — Modal (recommended; this is exactly how the checkpoint was made)

[Modal](https://modal.com) runs a single command in the cloud on an L4 GPU (~**$0.55** for the whole run).
Grab the code first:

```bash
# on your laptop (not Colab)
git clone https://github.com/VizuaraAI/RL-in-Production-Bootcamp-Resources
cd RL-in-Production-Bootcamp-Resources/phase-2/grasping-in-the-dark
pip install modal && modal token new        # one-time Modal auth
```

**Train** (detached, single L4). `save-freq` is deliberately small because cloud GPUs get preempted near
convergence — you want a good checkpoint on disk the instant it exists:
```bash
export HF_TOKEN=hf_...        # optional: auto-pushes each checkpoint to your Hub repo
modal run --detach modal/train_hilserl.py::main \
  --config configs/train_gym_hil.json \
  --save-freq 500 --gpu L4 \
  --hf-repo <your-hf-username>/hilserl-panda-pickcube-sac
```
**Watch it learn**, then render/evaluate a specific checkpoint:
```bash
modal run modal/train_hilserl.py::evaluate    --job-name hilserl_panda_pickcube --n-episodes 50
modal run modal/train_hilserl.py::render_demo  --job-name hilserl_panda_pickcube --ckpt last --n-try 12
# publish a checkpoint yourself (if you didn't pass --hf-repo above):
HF_TOKEN=hf_... modal run modal/train_hilserl.py::publish_main --hf-repo <you>/hilserl-panda-pickcube-sac
```
The config caps the run at 100k actor env-steps, but the policy **converges in ~5,000 gradient steps
(~40 min)** — its first success appears near step ~2,900. Watch the reward curve and stop when it
saturates. If a run gets preempted, re-launch the same command with `--resume` — completed checkpoints
are already on the Modal volume.

### Option 2 — your own GPU box (the raw LeRobot way, no Modal)

HIL-SERL is two processes that talk over `127.0.0.1:50051`. **Start the learner first**, then the actor in
a second terminal:
```bash
git clone https://github.com/huggingface/lerobot && cd lerobot
pip install -e '.[hilserl]'          # pulls gym-hil + mujoco + grpcio + the SAC stack

# terminal 1 — the learner (holds the replay buffers, does gradient updates):
python -m lerobot.rl.learner --config_path /path/to/configs/train_gym_hil.json

# terminal 2 — the actor (steps the env, streams transitions, receives fresh weights):
python -m lerobot.rl.actor  --config_path /path/to/configs/train_gym_hil.json
```
Use the **`configs/train_gym_hil.json`** from this repo — it's the upstream example with **one change**:
`env.task = "PandaPickCube-v0"` (autonomous & headless; the upstream default is the *gamepad* env, which
needs a display + controller).

### The config, decoded (the knobs that matter)

`configs/train_gym_hil.json` is a `TrainRLServerPipelineConfig`. The load-bearing fields:

| Field | Value | What it does |
|---|---|---|
| `env.task` | `PandaPickCube-v0` | autonomous, headless (no human / no input device) |
| `policy.type` | `gaussian_actor` | the SAC policy network |
| `algorithm.type` | `sac` | `num_critics: 2`, `utd_ratio: 2`, `discount: 0.97`, LayerNorm critics |
| `mixer` / `online_ratio` | `online_offline` / `0.5` | **RLPD's 50/50 sampling** |
| `dataset.repo_id` | `lilkm/pick_cube_franka_panda_30` | the offline seed demos (Section 2) |
| `steps` | `100000` | actor-step cap (you'll converge ~5k gradient steps in) |
| `save_freq` | lower to **500** | intermediate checkpoints (preemption insurance) |
| `policy.actor_learner_config` | `127.0.0.1:50051` | the gRPC channel between the two processes |

> **A teaching caveat we verified:** the paper's RLPD uses big ensembles (E=10) and a very high UTD (G=20).
> The LeRobot *sim* config uses the lighter `num_critics: 2`, `utd_ratio: 2` — plenty for this task. Teach
> E=10/G=20 as *the paper's* recipe; don't claim the sim config uses them.

### The four bugs you'll hit (we already hit them for you)

Distributed RL on ephemeral cloud GPUs is real engineering. Each of these is fixed in this repo's Modal
app — but if you run the raw LeRobot path, expect them:
1. **The output directory must be pristine.** LeRobot's config validator refuses to start if it already
   exists — and *both* learner and actor validate against the *same* dir. Give the actor its own throwaway
   output dir (it checkpoints nothing; it gets weights over gRPC).
2. **A checkpoint-serialization crash.** At each save the learner dumps the replay buffer to disk as a
   dataset; the image writer rejects `float32` pixels in `[0,255]` and halts training at the *first* save.
   Patch the writer to clip-and-cast to `uint8` (see `modal/patch_image_writer.py`).
3. **W&B auth even when offline.** Disable W&B when no key is present, or it blocks startup.
4. **Preemption near convergence.** Save every ~500 steps and push each checkpoint to the Hub *from inside*
   the job, so a good policy externalizes the instant it exists. (Our first 100% run was lost to
   `save_freq` being too high + a preemption — hence `save-freq 500`.)

#### (Optional, experimental) a tiny taste of training *inside* Colab

You cannot do a full run here, but you *can* watch the actor↔learner loop start and the buffer fill for a
few hundred steps. This is **off by default** (it launches two background processes and is fragile in a
headless runtime) — flip `RUN_MINI = True` only if you're curious. The real run is Option 1 or 2 above.

In [ ]:
RUN_MINI = False   # <- set True to launch a short in-Colab actor+learner (experimental)

if RUN_MINI:
    import subprocess, time, json, tempfile, os, signal
    # a throwaway config: tiny run, autonomous env, learner+actor on localhost
    base = json.load(open(f'{local}/train_config.json'))
    base['env']['task'] = 'PandaPickCube-v0'
    base['steps'] = 400
    base.setdefault('wandb', {})['enable'] = False
    lrn = tempfile.mkdtemp(); act = tempfile.mkdtemp()
    lcfg = os.path.join(lrn, 'cfg.json'); base['output_dir'] = os.path.join(lrn, 'out')
    json.dump(base, open(lcfg, 'w'))
    acfg = os.path.join(act, 'cfg.json'); base['output_dir'] = os.path.join(act, 'out')
    json.dump(base, open(acfg, 'w'))
    env2 = {**os.environ, 'MUJOCO_GL': 'egl', 'PYOPENGL_PLATFORM': 'egl'}
    L = subprocess.Popen(['python', '-m', 'lerobot.rl.learner', '--config_path', lcfg], env=env2)
    time.sleep(20)   # let the learner bind the gRPC port first
    A = subprocess.Popen(['python', '-m', 'lerobot.rl.actor',  '--config_path', acfg], env=env2)
    try:
        time.sleep(180); print('mini actor+learner ran for ~3 min — check the process output above')
    finally:
        for pr in (A, L):
            pr.send_signal(signal.SIGINT); time.sleep(2); pr.kill()
else:
    print('RUN_MINI is False — skipping the in-Colab mini training. Use Modal (Option 1) for a real run.')

## 8 · Take it to a real **SO-101** ($200 arm) — the bring-up guide (Path C)

The point of learning on a *Franka Panda in simulation* is a **real arm on your desk**. The encouraging
fact is how little of the *learning* code changes: the async actor–learner, the SAC learner, RLPD, and
the Gaussian-actor policy are **reused byte-for-byte**. What changes is the **robot/env config**, and what
you must **collect** is real demos + a reward signal (there's no simulator to hand you the reward now).

> **This runs on the robot's host machine, not in Colab.** Below is the concrete order of operations.

### The checklist

**1 · Hardware (~$200).** An **SO-101 follower** (6 Feetech servos on a serial bus) + a **leader** arm for
teleoperation + **two USB cameras** (a fixed front cam + a wrist cam). Assemble per the
[LeRobot SO-101 doc](https://huggingface.co/docs/lerobot/so101).

**2 · Install on the robot host.**
```bash
git clone https://github.com/huggingface/lerobot && cd lerobot
pip install -e '.[hilserl]'
```

**3 · Motors + calibration.** Set each servo's **id** and **baudrate**, then run the calibration flow from
the SO-101 doc (this maps raw encoder ticks → joint angles). Do this for **both** the follower and leader.

**4 · Define the workspace, then record ~15–25 teleop demos.** Drive the follower with the leader and
record successful grasps. **Tight end-effector workspace bounds are the single biggest lever on speed and
safety** — set them deliberately.
```bash
# record demos into a LeRobot dataset (repo_id becomes your offline seed set)
python -m lerobot.rl.gym_manipulator --config_path configs/so101_env.json   # mode: "record"
```

**5 · Collect + train the reward classifier.** On hardware there's no free reward, so you label a few
hundred camera frames success/failure and train a small classifier (LeRobot uses
[`helper2424/resnet10`](https://huggingface.co/helper2424/resnet10)):
```yaml
# in the train config (real-robot only — sim has no such block):
reward_model: { type: reward_classifier, model_name: helper2424/resnet10, model_type: cnn, num_cameras: 2, num_classes: 2 }
env.processor.reward_classifier: { pretrained_path: <your-classifier>, success_threshold: 0.7, success_reward: 1.0 }
```

**6 · Write the training config — the diff from sim.** Start from `configs/train_gym_hil.json` and change
only the robot/env layer:
- `env` → an `so101_follower` robot + your two cameras + a leader arm for teleop,
- end-effector-space **inverse kinematics** + the tight **workspace bounds** from step 4,
- add the `reward_model` / `reward_classifier` block from step 5,
- keep **`algorithm`, `policy`, `mixer`, `online_ratio` identical** — that's the reused learning brain.

**7 · Launch training *with* a human in the loop.** Same two processes as Section 7, but now the actor
runs the **gamepad** env so you can intervene when the policy is about to fail (those corrections enter the
buffer; your intervention rate falls to zero as it improves):
```bash
python -m lerobot.rl.learner --config_path configs/train_so101.json    # terminal 1
python -m lerobot.rl.actor  --config_path configs/train_so101.json    # terminal 2 (you hold the gamepad)
```

**8 · Evaluate + iterate.** Roll out the checkpoint; if it plateaus, add a few more demos or tighten the
workspace. Practitioners reach a working real SO-101 grasp in roughly **1–3 hours** this way.

### Safety + honest expectations

- **Safety:** keep a hand on the e-stop / power, cap end-effector velocity, and set conservative workspace
  bounds *before* the first autonomous rollout. A fresh policy moves unpredictably.
- **Numbers are order-of-magnitude, not benchmarks.** One careful public write-up
  ([ggando](https://ggando.com/blog/so101-hil-serl)) reports ~70% success after 757 episodes / ~3 h. Your
  mileage depends on cameras, lighting, and bounds.
- **Don't over-generalize others' negative results.** Indraneel Patil's honest
  [7-experiment write-up](https://indraneelpatil.github.io/blog/2026/hil-serl) found sim-pretraining didn't
  help *in his single setup* and that **tight workspace bounds are mandatory** — treat these as leads to
  test, not laws.

## 9 · Recap & reading

**What you did:** loaded a real SAC+RLPD policy, watched it grasp, measured its success rate, read the
learning curve as a *phase transition*, and saw exactly how to (B) reproduce the training and (C) put the
same stack on a real SO-101.

**The one-line takeaway:** from a reward it initially never receives, on a single cheap GPU, SAC + RLPD
teaches an arm to grasp in ~40 minutes — and the critic learns in the dark long before the actor ever scores.

### Reading (in dependency order)
- **SAC** — [arXiv:1801.01290](https://arxiv.org/abs/1801.01290) — the base max-entropy off-policy actor-critic.
- **RLPD** — [arXiv:2302.02948](https://arxiv.org/abs/2302.02948) — *why* it converges in an hour (50/50 sampling + LayerNorm critics).
- **SERL** — [arXiv:2401.16013](https://arxiv.org/abs/2401.16013) — the demos-only predecessor.
- **HIL-SERL** — [arXiv:2410.21845](https://arxiv.org/abs/2410.21845) · [project page](https://hil-serl.github.io) · [intro video](https://www.youtube.com/watch?v=GoJSW8e2qbI).
- **LeRobot HIL-SERL (sim)** — [docs](https://huggingface.co/docs/lerobot/hilserl_sim) — the basis for this notebook.
- **LeRobot HIL-SERL (real robot)** — [docs](https://huggingface.co/docs/lerobot/hilserl) — the full operational manual (config schema, workspace bounds, reward classifier, gRPC commands).
- **SO-101** — [setup doc](https://huggingface.co/docs/lerobot/so101) · [ggando recipe](https://ggando.com/blog/so101-hil-serl) · [Indraneel Patil's field notes](https://indraneelpatil.github.io/blog/2026/hil-serl).

**This project:** code, checkpoint, paper, and website → [https://github.com/VizuaraAI/RL-in-Production-Bootcamp-Resources/tree/main/phase-2/grasping-in-the-dark](https://github.com/VizuaraAI/RL-in-Production-Bootcamp-Resources/tree/main/phase-2/grasping-in-the-dark).

*Vizuara AI Labs · RL in Production — Session 3 (Robotics).*